In [0]:
dbutils.widgets.text("catalog", "claims_dev")
dbutils.widgets.text("through", "2026-01-31")

In [0]:
import os
import sys

sys.path.append(os.path.abspath(".."))

import pandas as pd
from pyspark.sql import functions as F
from src.generator import build

catalog = dbutils.widgets.get("catalog")
through = dbutils.widgets.get("through")
root = f"/Volumes/{catalog}/landing/raw"
data = build()

In [0]:
def written(path):
    try:
        return {f.name.strip("/").split("=")[1] for f in dbutils.fs.ls(path) if f.name.startswith("arrival_date=")}
    except Exception:
        return set()


def write_daily(pdf, path, fmt):
    new = pdf[(pdf["arrival_date"] <= through) & ~pdf["arrival_date"].isin(written(path))]
    if new.empty:
        return 0
    new = new.dropna(axis=1, how="all")
    sdf = spark.createDataFrame(new)
    for c, t in sdf.dtypes:
        if t == "double":
            sdf = sdf.withColumn(c, F.when(F.isnan(c), None).otherwise(F.col(c)))
    sdf.write.mode("append").partitionBy("arrival_date").format(fmt).save(path)
    return len(new)


def write_once(pdf, path, fmt):
    try:
        dbutils.fs.ls(path)
        return 0
    except Exception:
        spark.createDataFrame(pdf).write.format(fmt).save(path)
        return len(pdf)

In [0]:
ev = data["claim_events"]
inv = data["investigations"].assign(arrival_date=lambda d: d["investigated_on"].dt.strftime("%Y-%m-%d"))

counts = {
    "policies_cdc": write_daily(data["policies_cdc"], f"{root}/policies_cdc", "json"),
    "claim_events_app": write_daily(ev[ev["source"] == "app"], f"{root}/claim_events/app", "json"),
    "claim_events_tpa": write_daily(ev[ev["source"] == "tpa"], f"{root}/claim_events/tpa", "avro"),
    "investigations": write_daily(inv, f"{root}/investigations", "parquet"),
    "workshops": write_once(data["workshops"], f"{root}/workshops", "orc"),
    "coverages": write_once(data["coverages"], f"{root}/coverages", "parquet"),
}
counts

In [0]:
for p in ["policies_cdc", "claim_events/app", "claim_events/tpa", "investigations"]:
    days = sorted(written(f"{root}/{p}"))
    print(p, len(days), days[0] if days else None, days[-1] if days else None)